# U10 · La IA como copiloto de ciencia de datos — un recorrido end-to-end

> **Este es el cuaderno-síntesis del curso.** No aprende nada nuevo de estadística: recorre, de principio a fin, el mismo caso que ya viste por partes en las unidades 2 a 5 (`pacientes.csv` → un modelo de `evento_cv` evaluado con honestidad), pero contado como lo que **de verdad** vas a hacer en 2026: **dirigir un bucle** con un asistente de IA, en vez de teclear tú cada línea.

> ⚠️ Todos los datos son **sintéticos** (generados por código, semilla fija). **No representan pacientes reales.** La primera celda de código los crea sola: no hay que descargar nada.

**Cómo está organizado cada bloque de este cuaderno**, y por qué:

1. 🤖 **Una celda de cita con el PROMPT** que le darías tú, como profesional sanitario, a tu asistente de IA (Cowork, Claude Code, Codex, Manus…) en ese punto del proceso.
2. 💻 **El código que el asistente produciría y ejecutaría** — aquí ya resuelto, para que veas el resultado real sin esperar.
3. 🩺 **Una celda de interpretación clínica** de lo que acaba de pasar: qué significa el número o el gráfico, y qué decidimos a continuación.

Ese ritmo (prompt → ejecución → lectura clínica) **es** el bucle de evidencia de la sección 10.3 de la unidad. Tú no memorizas sintaxis: aprendes a **pedir bien** y a **leer con criterio** lo que vuelve.

[Abrir en Colab](https://colab.research.google.com/drive/1Nz_xdzwz0SX_fOEbzSl0jbVLVHtUnu5H)


## 0. Preparamos los datos (esta celda se ejecuta sola)

Como en todo el curso, la primera celda de código genera los ficheros sintéticos en la carpeta de trabajo si no están ya ahí. Ejecútala (▶ o *Mayús + Enter*) y sigue hacia abajo.


In [ ]:
# === Generación de los datos sintéticos del curso (se ejecuta sola) ===
# TODOS LOS DATOS SON SINTÉTICOS. No representan pacientes reales. Semilla fija -> reproducible.
import os, numpy as np, pandas as pd

def generar_datos_clinicos(carpeta="."):
    """Crea los CSV del curso si no están ya en la carpeta. Devuelve la ruta."""
    if os.path.exists(os.path.join(carpeta, "pacientes.csv")):
        return carpeta
    RNG = np.random.default_rng(2026)   # misma semilla en todo el curso

    # --- Centros ---
    AREAS = ["Valencia","Alicante","Castellón","Madrid","Barcelona","Sevilla","Bilbao","Zaragoza"]
    nc = 24
    tipo = RNG.choice(["hospital","centro de salud"], nc, p=[0.35,0.65])
    camas = np.where(tipo=="hospital", RNG.integers(120,900,nc), RNG.integers(0,25,nc))
    nserv = np.where(tipo=="hospital", RNG.integers(12,40,nc), RNG.integers(3,10,nc))
    centros = pd.DataFrame({"centro_id":[f"C{i:03d}" for i in range(1,nc+1)],"tipo":tipo,
        "area":RNG.choice(AREAS,nc),"camas":camas,"n_servicios":nserv,
        "urgencias_dia_media":np.where(tipo=="hospital",RNG.integers(80,320,nc),RNG.integers(5,40,nc)),
        "ratio_mayores65":RNG.uniform(0.12,0.34,nc).round(3)})
    centros.to_csv(os.path.join(carpeta,"centros.csv"), index=False)

    # --- Pacientes (tabla central) ---
    N = 20000
    sexo = RNG.choice(["M","F"], N, p=[0.49,0.51])
    edad = np.clip(np.where(RNG.random(N)<0.6, RNG.normal(58,15,N), RNG.normal(40,12,N)),18,95).round().astype(int)
    p_act = np.clip(0.28-0.0016*(edad-18),0.06,0.28); p_ex = np.clip(0.10+0.004*(edad-18),0.10,0.40); p_nu = 1-p_act-p_ex
    tabaquismo = np.array([RNG.choice(["nunca","ex","activo"],p=[a,b,c]) for a,b,c in zip(p_nu,p_ex,p_act)])
    actividad = RNG.choice(["baja","media","alta"], N, p=[0.4,0.4,0.2])
    antec = RNG.binomial(1,0.22,N)
    actn = pd.Series(actividad).map({"baja":1.5,"media":0.0,"alta":-1.3}).values
    imc = np.clip(RNG.normal(26.5,4.2,N)+0.03*(edad-50)+actn+RNG.normal(0,1.0,N),15,48).round(1)
    fa = (tabaquismo=="activo").astype(float); fe = (tabaquismo=="ex").astype(float)
    ta_s = np.clip(RNG.normal(120,12,N)+0.45*(edad-45)+0.7*(imc-25)+6*fa+RNG.normal(0,6,N),85,220).round().astype(int)
    ta_d = np.clip(0.55*ta_s+RNG.normal(20,6,N),50,130).round().astype(int)
    glu = np.clip(RNG.normal(100,12,N)+1.0*(imc-25)+0.3*(edad-45)+9*antec+RNG.normal(0,7,N),60,320).round(1)
    diabetes = (glu>126).astype(int)
    col = np.clip(RNG.normal(195,30,N)+0.4*(edad-45)+1.1*(imc-25)+RNG.normal(0,15,N),110,380).round(1)
    hdl = np.clip(RNG.normal(55,12,N)-0.25*(imc-25)+6*(sexo=="F")-5*fa-3*actn+RNG.normal(0,6,N),20,110).round(1)
    z = (-3.1+0.062*(edad-50)+0.019*(ta_s-120)+0.008*(col-190)-0.028*(hdl-55)+0.055*(imc-25)
         +0.85*fa+0.30*fe+0.60*diabetes+0.45*antec+0.55*(sexo=="M")
         +0.012*np.maximum(edad-65,0)*fa+RNG.normal(0,0.35,N))
    p = 1/(1+np.exp(-z))
    riesgo = (100*p).round(1); evento = RNG.binomial(1,p)
    pacientes = pd.DataFrame({"paciente_id":[f"P{i:05d}" for i in range(1,N+1)],"edad":edad,"sexo":sexo,
        "imc":imc,"ta_sistolica":ta_s,"ta_diastolica":ta_d,"glucemia":glu,"colesterol_total":col,"hdl":hdl,
        "tabaquismo":tabaquismo,"actividad_fisica":actividad,"antecedentes_familiares":antec,
        "diabetes":diabetes,"riesgo_cv_10a":riesgo,"evento_cv":evento})
    pacientes.to_csv(os.path.join(carpeta,"pacientes.csv"), index=False)

    # --- Versión "sucia" para EDA ---
    d = pacientes.copy()
    d["sexo"] = d["sexo"].map(lambda s: RNG.choice({"M":["M","m","Masculino","H","Hombre"],"F":["F","f","Femenino","Mujer"]}[s]))
    idx = RNG.choice(d.index,int(N*0.06),replace=False)
    d["glucemia"] = d["glucemia"].astype(object)   # permite mezclar texto (mmol/L con coma) y números
    d.loc[idx,"glucemia"] = [str(round(v/18.0,1)).replace(".",",") for v in d.loc[idx,"glucemia"]]
    prob_na = np.where(pacientes["edad"]<40,0.12,0.04); mask = RNG.random(N)<prob_na
    d.loc[mask,"hdl"] = np.nan
    d.loc[RNG.choice(d.index,int(N*0.05),replace=False),"colesterol_total"] = np.nan
    d.loc[RNG.choice(d.index,25,replace=False),"edad"] = RNG.integers(150,260,25)
    d.loc[RNG.choice(d.index,20,replace=False),"ta_sistolica"] = -RNG.integers(1,90,20)
    d.loc[RNG.choice(d.index,15,replace=False),"imc"] = RNG.uniform(80,200,15).round(1)
    d["colesterol_total"] = d["colesterol_total"].astype(object); d["ta_diastolica"] = d["ta_diastolica"].astype(object)
    d.loc[RNG.choice(d.index,30,replace=False),"colesterol_total"] = "desconocido"
    d.loc[RNG.choice(d.index,20,replace=False),"ta_diastolica"] = "N/D"
    d["tabaquismo"] = d["tabaquismo"].map(lambda s: RNG.choice({"nunca":["nunca","No fumador","NUNCA"],
        "ex":["ex","exfumador","Ex-fumador"],"activo":["activo","Fumador","SI"]}[s]))
    d = pd.concat([d, d.sample(400,random_state=3)], ignore_index=True)
    idx = RNG.choice(d.index,200,replace=False); d.loc[idx,"paciente_id"] = d.loc[idx,"paciente_id"].map(lambda s:" "+s+" ")
    d = d.sample(frac=1,random_state=11).reset_index(drop=True)
    d.to_csv(os.path.join(carpeta,"pacientes_sucio.csv"), index=False)

    # --- Urgencias (serie temporal) ---
    ND = 730; fechas = pd.date_range("2024-01-01", periods=ND, freq="D")
    doy = fechas.dayofyear.values; dow = fechas.dayofweek.values
    fest_set = {(1,1),(1,6),(5,1),(8,15),(10,12),(11,1),(12,6),(12,8),(12,25)}
    festivo = np.array([1 if (f.month,f.day) in fest_set else 0 for f in fechas])
    gripe = np.array([1 if f.month in (12,1,2) else 0 for f in fechas])
    temp = (15+10*np.sin(2*np.pi*(doy-110)/365)+RNG.normal(0,2.5,ND)).round(1)
    est = 1+0.14*np.sin(2*np.pi*(doy-20)/365)
    sem = np.where(dow==0,1.18,np.where(dow>=5,1.10,1.0))
    mu = 120.0*est*sem*(1+0.22*gripe)*(1+0.15*festivo)
    ingresos = RNG.poisson(np.maximum(mu,1)).astype(int)
    pd.DataFrame({"fecha":fechas.strftime("%Y-%m-%d"),"ingresos":ingresos,"festivo":festivo,
        "temporada_gripe":gripe,"temperatura":temp}).to_csv(os.path.join(carpeta,"urgencias_diarias.csv"), index=False)

    # --- Notas clínicas (texto) ---
    plant = {"cardiología":["dolor torácico opresivo de {t} de evolución, irradiado a brazo izquierdo",
        "disnea de esfuerzo progresiva y palpitaciones, antecedente de hipertensión",
        "control de anticoagulación, fibrilación auricular conocida, sin dolor actual",
        "edemas en miembros inferiores y ortopnea, sospecha de insuficiencia cardiaca"],
      "respiratorio":["tos productiva de {t}, fiebre y disnea, auscultación con crepitantes",
        "sibilancias y opresión torácica, antecedente de asma, mala respuesta a inhalador",
        "disnea súbita y dolor pleurítico, saturación disminuida",
        "tos seca persistente y febrícula, contacto con caso respiratorio"],
      "digestivo":["dolor abdominal en fosa iliaca derecha de {t}, náuseas y febrícula",
        "epigastralgia y pirosis, relación con las comidas, sin signos de alarma",
        "diarrea de {t} sin productos patológicos, buen estado general",
        "ictericia y coluria, dolor en hipocondrio derecho"],
      "neurología":["cefalea intensa de inicio súbito, la peor de su vida, con fotofobia",
        "focalidad neurológica de {t}, debilidad en hemicuerpo y disartria",
        "mareo con giro de objetos y vómitos, sin cortejo vegetativo",
        "crisis comicial presenciada, recuperación progresiva del nivel de conciencia"],
      "traumatología":["caída casual con dolor e impotencia funcional en muñeca, deformidad",
        "lumbalgia mecánica de {t} tras esfuerzo, sin déficit neurológico",
        "esguince de tobillo con edema y dificultad para la deambulación",
        "gonalgia y derrame articular tras traumatismo deportivo"]}
    prio = {"cardiología":[0.35,0.45,0.20],"respiratorio":[0.30,0.45,0.25],"digestivo":[0.20,0.50,0.30],
        "neurología":[0.45,0.35,0.20],"traumatología":[0.12,0.48,0.40]}
    tiempos = ["horas","un día","dos días","una semana","varios días"]; esp_list = list(plant.keys()); rows=[]
    for _ in range(3000):
        e = RNG.choice(esp_list); txt = RNG.choice(plant[e]).replace("{t}",RNG.choice(tiempos))
        rows.append((txt,e,RNG.choice(["alta","media","baja"],p=prio[e]),RNG.choice(centros["centro_id"].values)))
    pd.DataFrame(rows,columns=["texto","especialidad","prioridad","centro_id"]).to_csv(os.path.join(carpeta,"notas_clinicas.csv"),index=False)

    # --- Wearable (señal) ---
    rows=[]
    for s in range(1,201):
        fcb=RNG.normal(66,6); pb=RNG.normal(7500,2500); sb=RNG.normal(7.0,0.8); anom=RNG.random()<0.05
        for dia in range(1,31):
            fc=fcb+RNG.normal(0,3); pasos=max(0,pb+RNG.normal(0,1500)); sue=float(np.clip(sb+RNG.normal(0,0.6),3,11))
            if anom and dia in (14,15,16): fc+=RNG.uniform(18,30); sue-=RNG.uniform(1.5,3)
            rows.append((f"S{s:03d}",dia,round(fc,1),int(pasos),round(sue,1)))
    pd.DataFrame(rows,columns=["sujeto_id","dia","fc_reposo","pasos","horas_sueno"]).to_csv(os.path.join(carpeta,"wearable.csv"),index=False)
    return carpeta

generar_datos_clinicos(".")
print("Datos sintéticos listos en la carpeta actual. (Recuerda: son datos SINTÉTICOS, no reales.)")

## 1. El método: no programas, **diriges un bucle**

Antes de la primera línea de código, conviene fijar la idea que sostiene todo el cuaderno. Durante décadas, hacer ciencia de datos era un **guion lineal**: explorar, limpiar, entrenar, evaluar, escribir el informe — paso a paso, a mano. Con un asistente de IA capaz de **ejecutar código de verdad**, el trabajo cambia de forma: tú fijas el **objetivo, el contexto clínico y el criterio de éxito**; el asistente **propone, ejecuta, mide e interpreta**; y tú **validas y decides** si esa vuelta se queda o se descarta. Y se repite. A esta forma de trabajar se le llama **_loop engineering_**: no escribes el pipeline, **diriges un bucle de evidencia**.

{% hint style="success" %}
**💡 Idea clave**

El cambio de oficio en una frase: **antes escribías el pipeline; ahora diriges un bucle**. Tú aportas la **pregunta clínica correcta** y el **juicio**; el asistente aporta **velocidad y amplitud**. Lo que **nunca** delegas es la validación: en salud, **cada vuelta** del bucle la revisa un criterio clínico.
{% endhint %}

Vamos a recorrer **seis pasos** de ese bucle, aplicados al mismo caso de las unidades 2 a 5: de `pacientes.csv` a un modelo de `evento_cv` evaluado con honestidad. En cada paso verás primero el **prompt** (lo que tú escribirías) y después el **código y el resultado** (lo que el asistente produciría y ejecutaría de verdad).


## 2. Antes de arrancar: le pedimos que **critique nuestro encargo**

El primer gesto de un buen copiloto no es "ejecutar ya", es **pensar contigo primero**. A esto lo llamamos *metaprompting*: pedirle al propio asistente que revise tu prompt inicial, señale lo que falta (métrica, validación, fugas) y te proponga una versión mejor **antes** de tocar una sola celda.


> 🤖 **PROMPT — Metaprompting antes de ejecutar**
>
> *"Antes de ejecutar nada, actúa como un experto en ciencia de datos clínicos y critica este encargo mío: quiero predecir `evento_cv` en `pacientes.csv` (datos sintéticos) para priorizar a quién citar antes en prevención cardiovascular. Dime: (1) qué información clave me falta, (2) si la métrica 'accuracy' que se me había ocurrido encaja con mi decisión clínica o me propones otra, (3) qué riesgos de fuga de datos ves. Luego resume el plan ya corregido."*


**Lo que respondería el asistente (resumido, es la parte que no se «ejecuta», se razona):**

- *Información que falta*: en qué horizonte temporal se define `evento_cv`, qué columnas existirían realmente en el momento de decidir a quién citar (¿la fecha del evento estaría disponible? no debería), y cuál de los dos errores —no detectar a alguien que sí tendrá el evento (falso negativo) o citar de más a alguien sano (falso positivo)— nos duele más.
- *Sobre la métrica*: con una prevalencia de `evento_cv` ≈ 19 %, la **exactitud (accuracy) es una mala brújula** — un modelo que dijera "nadie tendrá evento" acertaría el 81 % de las veces y sería clínicamente inútil (esto ya lo viste en la U3). Propone usar **AUC-ROC** como métrica de comparación entre modelos, y añadir sensibilidad/recall como lectura clínica del umbral final.
- *Riesgos de fuga*: vigilar que no se use ninguna variable que solo exista **después** del evento, y que la partición se haga **por paciente**, nunca mezclando información de train y test.

Este intercambio no ha ejecutado ni una línea de código, pero ha evitado el error más caro de todo el proyecto: **medir con la métrica equivocada**. Es la razón de que este bloque vaya *antes* del bucle: metaprompting primero, ejecución después.


## 3. Paso 1 del bucle · "Explora y limpia estos datos"

Con el encargo ya afinado, arrancamos el bucle de verdad. La primera vuelta nunca es modelar: es **mirar los datos con calma**, exactamente lo que aprendiste en la U2.


> 🤖 **PROMPT — Paso 1: EDA rápido**
>
> *"Te paso `pacientes.csv` (cohorte SINTÉTICA, 20 000 pacientes). Antes de modelar nada, hazme una exploración rápida: forma de la tabla, tipos de columna, huecos (nulos), un `describe()` de las variables numéricas, y la prevalencia de `evento_cv`. Dime si ves algo raro."*


In [ ]:
import pandas as pd

pacientes = pd.read_csv("pacientes.csv")

print("Forma de la tabla (filas, columnas):", pacientes.shape)
print()
pacientes.head()

**Lectura clínica:** 20 000 filas, una por paciente, con las variables clínicas habituales (edad, tensión, glucemia, colesterol, hábitos…) y, al final, los dos objetivos del curso: `riesgo_cv_10a` (%, para regresión) y `evento_cv` (0/1, para clasificación). Nada raro a simple vista; seguimos mirando huecos y tipos.


In [ ]:
print("Huecos (nulos) por columna:")
print(pacientes.isna().sum().to_string())
print()
print("Tipos de dato por columna:")
print(pacientes.dtypes.to_string())

**Lectura clínica:** cero nulos y los tipos son los esperados (números para las medidas clínicas, texto para sexo/tabaquismo/actividad). Esta cohorte viene ya "limpia" porque es sintética y de curso; en la práctica real este paso es donde se descubren la mayoría de los sustos (huecos, formatos mezclados, valores imposibles) — lo viste con detalle en la U2 usando `pacientes_sucio.csv`. Aquí, con la versión limpia, seguimos al resumen numérico.


In [ ]:
pacientes[["edad","imc","ta_sistolica","ta_diastolica","glucemia",
           "colesterol_total","hdl","riesgo_cv_10a"]].describe().round(1)

**Lectura clínica:** los rangos son clínicamente plausibles — edades entre 18 y 95, tensión sistólica en torno a 120, glucemia media ~103 mg/dL. Nada fuera de lo razonable (ni tensiones negativas, ni edades de 300 años, que sí aparecían en la versión "sucia" de la U2). Con la tabla en orden, miramos el dato más importante antes de modelar: **cuántos pacientes tienen el evento**.


In [ ]:
import matplotlib.pyplot as plt

prevalencia = pacientes["evento_cv"].mean()
conteo = pacientes["evento_cv"].value_counts().sort_index()

plt.figure(figsize=(6, 4))
plt.bar(["Sin evento (0)", "Con evento (1)"], conteo.values,
        color=["#9fb3c8", "#e8663a"], edgecolor="white")
plt.ylabel("Nº de pacientes")
plt.title("Reparto de evento_cv (datos sintéticos)")
for i, v in enumerate(conteo.values):
    plt.text(i, v, f"{v:,}", ha="center", va="bottom")
plt.show()
print(f"Prevalencia de evento_cv: {prevalencia:.1%}")

**Lectura clínica — y esta es la que gobierna todo lo que viene:** solo **1 de cada 5 pacientes (≈19 %)** tiene el evento. Es un problema **desbalanceado**. Esto confirma lo que ya nos avisó el metaprompting del bloque anterior: la *accuracy* sería una métrica engañosa aquí, porque un modelo perezoso que dijera "nadie" acertaría el 81 % de las veces sin detectar a nadie útil. Con la exploración hecha, el bucle avanza a su siguiente vuelta: **plantear el problema con precisión**.


## 4. Paso 2 del bucle · "Plantea el problema y una partición honesta"

Ya sabemos qué hay en la tabla y cómo se reparte el objetivo. Toca la decisión que más fugas evita en todo el curso: **cómo separamos entrenamiento y test** sin que se "vea" información de más.


> 🤖 **PROMPT — Paso 2: formalizar el problema y partir sin fugas**
>
> *"Formaliza el problema: `evento_cv` es el target (clasificación binaria), una fila = un paciente, y ninguna columna de esta tabla existe 'después' del evento (verifica que no cuele nada raro). Separa entrenamiento y test de forma honesta — estratificado por `evento_cv` para conservar la prevalencia en ambos lados — y monta el preprocesado (escalado + codificación) DENTRO de un pipeline, para que se ajuste solo con el entrenamiento."*


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline

# Variables numéricas y categóricas, según el esquema del curso.
# Ninguna existe "después" del evento: todas estarían disponibles en el momento de decidir.
num = ["edad", "imc", "ta_sistolica", "ta_diastolica", "glucemia",
       "colesterol_total", "hdl", "antecedentes_familiares", "diabetes"]
cat = ["sexo", "tabaquismo", "actividad_fisica"]

X = pacientes[num + cat]
y = pacientes["evento_cv"]

# El preprocesado va DENTRO del pipeline: se ajustará SOLO con el train (sin fuga).
pre = ColumnTransformer([
    ("num", StandardScaler(), num),
    ("cat", OneHotEncoder(handle_unknown="ignore"), cat),
])

# Partición estratificada: conserva la prevalencia (~19%) en train y en test.
Xtr, Xte, ytr, yte = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y)

print(f"Entrenamiento: {len(Xtr):>6} pacientes  | prevalencia: {ytr.mean():.1%}")
print(f"Test (intacto): {len(Xte):>5} pacientes  | prevalencia: {yte.mean():.1%}")

**Lectura clínica:** hemos partido la cohorte en 15 000 pacientes para entrenar y 5 000 que quedan **reservados** con la misma proporción de eventos (~19 %) en ambos lados. A partir de aquí el **test es sagrado**: no lo va a tocar ninguna comparación, ninguna búsqueda de hiperparámetros. Solo lo miraremos **una vez**, al final, en el paso 6. Este es exactamente el tipo de decisión que **tú** debes vigilar en cualquier respuesta del asistente: si alguna vez "ajusta" algo mirando el test, hay fuga.


## 5. Paso 3 del bucle · "Prueba varios modelos y compáralos por AUC"

Con la partición honesta lista, pedimos amplitud: que el asistente entrene **varias familias de modelo** y las compare con la **misma vara de medir** en el mismo test.


> 🤖 **PROMPT — Paso 3: comparar familias de modelos**
>
> *"Entrena regresión logística, Random Forest y HistGradientBoosting sobre el mismo entrenamiento, y compáralas por AUC-ROC en el mismo test. Dame una tabla ordenada de mejor a peor."*


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score

modelos = {
    "Regresión logística": Pipeline([("pre", pre), ("clf", LogisticRegression(max_iter=2000))]),
    "Random Forest":        Pipeline([("pre", pre), ("clf", RandomForestClassifier(n_estimators=300, n_jobs=-1, random_state=0))]),
    "HistGradientBoosting": Pipeline([("pre", pre), ("clf", HistGradientBoostingClassifier(random_state=0))]),
}

probs_test = {}   # guardamos las probabilidades del test para reutilizarlas en pasos siguientes
filas = []
for nombre, modelo in modelos.items():
    modelo.fit(Xtr, ytr)
    p = modelo.predict_proba(Xte)[:, 1]
    probs_test[nombre] = p
    filas.append((nombre, roc_auc_score(yte, p)))

tabla_auc = (pd.DataFrame(filas, columns=["Modelo", "AUC en test"])
             .sort_values("AUC en test", ascending=False)
             .reset_index(drop=True))
tabla_auc.round(3)

**Lectura clínica:** las tres familias quedan **muy juntas**, con la **regresión logística** —la más simple e interpretable— igualando o ganando, con un AUC en torno a **0,84**. Esto es la lección de oro de las unidades 4-5 en directo: cuando el riesgo se comporta de forma aproximadamente log-aditiva, los modelos complejos **no compran ventaja real**. Un asistente sin este criterio podría quedarse con el modelo "más sofisticado" por costumbre; tu trabajo es exigir que **la tabla mande**, no la reputación del algoritmo.


## 6. Paso 4 del bucle · "Valida con validación cruzada y ajusta hiperparámetros"

Una sola partición train/test puede ser afortunada o desafortunada según qué pacientes cayeron en cada lado. Pedimos una estimación más sólida y, ya que estamos, afinar el candidato con más margen de mejora.


> 🤖 **PROMPT — Paso 4: CV y una búsqueda de hiperparámetros pequeña**
>
> *"Repite la comparación con validación cruzada estratificada de 5 pliegues (media y desviación entre pliegues, para saber cuánto fiarnos). Después, afina el Random Forest con una rejilla PEQUEÑA de hiperparámetros usando GridSearchCV, siempre sobre el entrenamiento — el test no se toca."*


In [ ]:
from sklearn.model_selection import cross_val_score, StratifiedKFold
import numpy as np

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)

for nombre, modelo in modelos.items():
    scores = cross_val_score(modelo, X, y, cv=cv, scoring="roc_auc", n_jobs=1)
    print(f"{nombre:22s} AUC por pliegue: {np.round(scores, 3)}  "
          f"media={scores.mean():.3f}  (±{scores.std():.3f})")

**Lectura clínica:** cada modelo da **cinco** valores de AUC (uno por pliegue); la desviación pequeña dice que son **estables** (no dependen de qué pacientes tocaron por azar), y las medias confirman —con más solidez que un solo split— lo que ya vimos: la logística compite de tú a tú con los ensembles. Esta es la cifra que usaríamos para decidir con honestidad.


In [ ]:
from sklearn.model_selection import GridSearchCV

# Rejilla DELIBERADAMENTE pequeña: 2 x 2 = 4 combinaciones.
rejilla = {
    "clf__n_estimators": [150, 300],
    "clf__max_depth":    [8, None],
}

busqueda = GridSearchCV(
    Pipeline([("pre", pre), ("clf", RandomForestClassifier(n_jobs=1, random_state=0))]),
    rejilla,
    cv=StratifiedKFold(n_splits=3, shuffle=True, random_state=0),
    scoring="roc_auc",
    n_jobs=-1,
)
busqueda.fit(Xtr, ytr)   # SOLO con el entrenamiento; el test sigue intacto

print("Mejor combinación de hiperparámetros:", busqueda.best_params_)
print("AUC de validación de esa combinación:", round(busqueda.best_score_, 3))

**Lectura clínica:** `GridSearchCV` probó las 4 combinaciones y se quedó con la mejor, evaluada siempre **dentro del entrenamiento**. Ese `best_score_` es optimista por construcción — es una cifra de validación, no la del test —, así que la única que contará de verdad es la del test reservado, que miraremos **una sola vez** en el paso 6. Aquí es donde un asistente sin supervisión podría "vender" el `best_score_` como el resultado final: **no lo es**, y corregir esa confusión es justo el tipo de validación clínica que te toca a ti.


## 7. Paso 5 del bucle · "Elige el mejor y explícame por qué predice lo que predice"

Con los números sobre la mesa, toca decidir con criterio clínico — no por moda — y pedir que la caja deje de ser negra: **¿en qué se fija el modelo elegido?**


> 🤖 **PROMPT — Paso 5: elegir y explicar con permutation_importance**
>
> *"Compara el Random Forest afinado con la regresión logística simple por AUC. Si no hay ventaja clara, razona por qué te quedarías con la más simple. Y para el modelo que elijamos, calcula la importancia de variables por permutación (permutation_importance) sobre el test, para saber en qué se apoya realmente."*


In [ ]:
modelo_final = busqueda.best_estimator_          # RF con los mejores hiperparámetros
prob_rf = modelo_final.predict_proba(Xte)[:, 1]

auc_rf  = roc_auc_score(yte, prob_rf)
auc_log = roc_auc_score(yte, probs_test["Regresión logística"])
print(f"AUC en test · Random Forest afinado : {auc_rf:.3f}")
print(f"AUC en test · Regresión logística    : {auc_log:.3f}")

**Lectura clínica:** el Random Forest afinado da un AUC **muy parecido** al de la logística simple. Todo el esfuerzo de búsqueda de hiperparámetros **no ha comprado una ventaja real**. Con criterio clínico, nos quedamos con la **regresión logística**: rinde igual y además llega con interpretabilidad de regalo (sus coeficientes se leen directamente como *odds ratio*). Es la elección que usaremos en el paso final.


In [ ]:
from sklearn.inspection import permutation_importance

modelo_elegido = modelos["Regresión logística"]   # la que elegimos por criterio clínico

r = permutation_importance(
    modelo_elegido, Xte, yte,
    scoring="roc_auc", n_repeats=5, random_state=0, n_jobs=-1)

importancias = (pd.DataFrame({
        "variable": (num + cat),
        "caida_de_AUC": r.importances_mean,
    })
    .sort_values("caida_de_AUC", ascending=False)
    .reset_index(drop=True))
importancias.round(4)

**Lectura clínica:** `permutation_importance` baraja al azar cada columna, una a una, y mide **cuánto se hunde el AUC**; si al estropear una variable el modelo empeora mucho, esa variable era importante de verdad. Aquí manda la **edad**, seguida de **tabaquismo**, **tensión sistólica** y **glucemia** — justo los grandes motores clínicos del riesgo cardiovascular. Que el modelo se apoye en las variables **correctas** es, en sí mismo, una primera validación: si arriba apareciera un identificador o una columna sin sentido clínico, sería una señal de alarma de fuga de datos.


In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
top = importancias.iloc[::-1]
ax.barh(top["variable"], top["caida_de_AUC"], color="#00AEC7", edgecolor="white")
ax.set_xlabel("Importancia = caída de AUC al barajar la variable")
ax.set_title("¿En qué se fija el modelo elegido? (permutation importance)")
plt.tight_layout(); plt.show()

**Lectura clínica:** de un vistazo, la edad domina con claridad y detrás aparecen los factores de siempre en prevención cardiovascular. Esta es la respuesta que le pedíamos al asistente en el prompt: no solo "qué AUC tiene", sino **por qué predice lo que predice** — y la respuesta coincide con lo que un clínico esperaría, que es la mejor señal de que el modelo no está haciendo algo raro.


## 8. Paso 6 del bucle · "Evalúa en test y dame la lectura clínica"

Última vuelta. Miramos el test **una sola vez**, con el modelo ya elegido y sin retocar nada a partir de este resultado.


> 🤖 **PROMPT — Paso 6: evaluación final y lectura clínica**
>
> *"Con la regresión logística ya elegida, evalúala UNA sola vez en el test: dame el AUC final, la curva ROC, y —como no detectar un evento (falso negativo) es el error que más nos duele— fija un umbral que dé sensibilidad alta y muéstrame sensibilidad, especificidad, VPP y VPN en ese punto. Cierra con una lectura clínica de dos frases."*


In [ ]:
from sklearn.metrics import roc_curve, confusion_matrix

prob_final = probs_test["Regresión logística"]   # ya calculada en el paso 3, sobre el test
auc_final = roc_auc_score(yte, prob_final)

fpr, tpr, _ = roc_curve(yte, prob_final)
plt.figure(figsize=(6, 5))
plt.plot(fpr, tpr, color="#00AEC7", lw=2, label=f"Regresión logística (AUC={auc_final:.2f})")
plt.plot([0, 1], [0, 1], "--", color="gray", label="Azar (AUC=0,50)")
plt.xlabel("1 - especificidad (falsos positivos)")
plt.ylabel("Sensibilidad (verdaderos positivos)")
plt.title("Curva ROC final en el test (evaluación única)")
plt.legend(loc="lower right"); plt.tight_layout(); plt.show()
print(f"AUC final en test: {auc_final:.3f}")

**Lectura clínica:** la curva se despega con claridad de la diagonal del azar; un **AUC ≈ 0,84** es una capacidad de ordenación notable para un modelo tan simple. Esta cifra, a diferencia de las que vimos en los pasos 3 y 4, es la que **de verdad cuenta**: sale del test que nunca participó en ninguna comparación ni búsqueda.


In [ ]:
# El umbral se fija SOBRE EL ENTRENAMIENTO (no sobre el test) para no hacer trampa.
prob_tr = modelos["Regresión logística"].predict_proba(Xtr)[:, 1]
objetivo_sens = 0.85
umbral = np.quantile(prob_tr[ytr == 1], 1 - objetivo_sens)

pred = (prob_final >= umbral).astype(int)
tn, fp, fn, tp = confusion_matrix(yte, pred).ravel()
sens = tp / (tp + fn)
esp  = tn / (tn + fp)
vpp  = tp / (tp + fp)
vpn  = tn / (tn + fn)

print(f"Umbral de decisión (fijado en train): {umbral:.3f}")
print(f"Sensibilidad : {sens:.1%}  (de los que tendrán evento, cuántos detectamos)")
print(f"Especificidad: {esp:.1%}  (de los sanos, cuántos descartamos bien)")
print(f"VPP          : {vpp:.1%}  (si el modelo avisa, prob. real de evento)")
print(f"VPN          : {vpn:.1%}  (si el modelo NO avisa, prob. real de estar sano)")

**Lectura clínica final:** buscábamos ~85 % de sensibilidad y en el test se materializa algo menor —recordatorio de que las cifras siempre bajan un poco al pasar de train a test—, pero seguimos **detectando a la mayoría** de los pacientes que tendrán el evento. A cambio, el **VPP es modesto**: con una prevalencia del 19 %, muchos avisos del modelo serán falsas alarmas; el **VPN es muy alto**, así que cuando el modelo dice "tranquilo", acierta casi siempre. Traducido a la consulta: este modelo es útil como **herramienta de priorización de citas de prevención** —para decidir a quién llamar antes—, no como un diagnóstico. Y **dónde poner el umbral es una decisión clínica de coste**, no un 0,5 por defecto.


## 9. Tu plantilla de prompt reutilizable para análisis clínico

El recorrido anterior no fue una casualidad de seis prompts sueltos: siguió un patrón. Aquí tienes una **plantilla de arranque** que puedes copiar, rellenar y reutilizar en cualquier análisis clínico con un asistente de IA. Cambia lo que está entre `[...]` por tu caso.


> 🤖 **PLANTILLA REUTILIZABLE — Copiloto de ciencia de datos clínicos**
>
> ```
> Actúa como mi copiloto de ciencia de datos clínicos y trabajaremos EN BUCLE.
> Conduces tú la ejecución; yo, como profesional sanitario, superviso y valido.
>
> CONTEXTO
> · Datos: [p. ej. pacientes.csv — cohorte SINTÉTICA, no son pacientes reales].
> · Una fila representa: [p. ej. un paciente]. Clave: [p. ej. paciente_id].
> · Decisión clínica que se tomará con la predicción: [p. ej. priorizar a quién
>   citar antes en prevención].
> · Objetivo a predecir (target): [p. ej. evento_cv, 0/1, prevalencia ≈19%],
>   definido como [ventana / cómo se observa].
> · Coste de los errores: me duele más [p. ej. un falso negativo] que [un falso
>   positivo].
> · PROHIBIDO usar (no disponible al decidir / fuga de datos): [columnas
>   posteriores al evento, identificadores sospechosos…].
>
> LO QUE TE PIDO
> 1. METAPROMPTING primero: dime qué información CLAVE me falta y si la métrica
>    encaja con mi decisión; propónme TÚ la métrica y la validación más
>    adecuadas y JUSTIFÍCALAS.
> 2. Luego repite este BUCLE hasta que propongas parar:
>    · PROPÓN  — la hipótesis o el paso que vas a probar y por qué.
>    · EJECUTA — escribe y CORRE el código (imputa/escala SOLO en train); si
>      falta una librería, instálala; si te falta una decisión clínica,
>      PREGÚNTAME.
>    · MIDE    — con la métrica acordada; enséñame el número o el gráfico REAL.
>    · INTERPRETA — qué significa clínicamente; ¿tiene sentido?
>    · DECIDE  — ¿conservas o descartas? Una frase de porqué y a la siguiente
>      vuelta.
> 3. DETENTE a pedir mi visto bueno en las decisiones importantes: métrica
>    final, exclusión de una variable, modelo elegido, umbral.
>
> REGLAS DE ORO (clínicas)
> · Empieza por un baseline y por lo simple; no aceptes una mejora sin
>   compararla.
> · No uses el test final para ajustar; evalúalo UNA sola vez, al final.
> · Nada de fuga de datos; todo el preprocesado, dentro del pipeline.
> · Un resultado "demasiado bueno" es sospechoso: búscame la fuga antes de
>   celebrar.
> · Cada afirmación, respaldada por una métrica EJECUTADA (no por tu
>   impresión).
> · Si crees que el problema está mal planteado o que el modelo NO debería
>   usarse en clínica, dímelo con argumentos.
>
> Empieza por tus preguntas y por proponerme la métrica y la validación.
> ```


## 10. El asistente puede equivocarse — por eso el humano valida

Con un copiloto tan capaz es tentador relajar la guardia. Es justo al revés: cuanto más autónomo el bucle, **más importa** vigilarlo. Hay tres riesgos concretos que ningún asistente detecta solo, y que solo tu criterio clínico puede atrapar.


{% hint style="warning" %}
**⚠️ Aviso · Tres riesgos que el asistente no ve solo**

- **Alucinaciones.** El modelo puede *afirmar con total aplomo* algo falso: citar una métrica que no calculó de verdad, o "recordar" un resultado que no ejecutó. Un número dicho con seguridad **no es un número medido** — exígele siempre la celda de código que lo produjo, como hemos hecho en todo este cuaderno.
- **Resultados plausibles pero erróneos.** El más peligroso, porque **no chirría**. Un AUC de 0,97 puede parecer excelente y ser el síntoma de una **fuga de datos**. En clínica, un modelo plausible-pero-falso es más peligroso que uno obviamente roto, porque **te lo crees**.
- **Fugas de datos que el asistente no ve.** No conoce tu flujo asistencial: no sabe que una columna se rellena *después* de que ocurra el evento, o que un identificador de centro está correlacionado con el desenlace por un motivo puramente organizativo. Esa fuga es invisible en la tabla y solo evidente con **tu** conocimiento del proceso.

Por eso, en cada vuelta de este cuaderno, la celda de código fue seguida de una celda de **lectura clínica**: ese es el lugar donde tú, y no el asistente, firmas el resultado.
{% endhint %}


## 11. Qué hemos aprendido

- **El recorrido completo, de un vistazo:** exploramos y limpiamos (paso 1) → planteamos el problema con una partición sin fugas (paso 2) → comparamos varios modelos por AUC (paso 3) → validamos con CV y afinamos hiperparámetros sin tocar el test (paso 4) → elegimos el mejor y explicamos por qué predice lo que predice (paso 5) → evaluamos una sola vez en el test y leímos el resultado en clínico (paso 6).
- **El desenlace fue la lección de oro del curso, con datos delante:** la humilde **regresión logística** igualó o superó a modelos más complejos en `evento_cv` (AUC ≈ 0,84), porque el riesgo cardiovascular en esta cohorte es aproximadamente log-aditivo. **No se adivina, se mide.**
- **Cada vuelta empezó con un prompt y terminó con una validación clínica.** Ese ritmo —pedir con contexto, ejecutar de verdad, interpretar en tu idioma, decidir tú— es *loop engineering*, y es la destreza que sustituye a "programar de memoria".
- **La plantilla reutilizable (sección 9)** te sirve para cualquier análisis nuevo: pon el contexto, pide metaprompting primero, deja que el asistente proponga-ejecute-mida-interprete-decida en bucle, y reserva para ti las decisiones que importan.
- **El humano valida siempre**, porque hay riesgos que el asistente no ve solo: alucinaciones, resultados plausibles-pero-falsos, y fugas de datos que solo tu conocimiento del proceso asistencial puede detectar.

### 🤖 Y para reproducir TODO este cuaderno de una sola vez

Si quisieras pedirle a un asistente que reprodujera este recorrido completo desde cero, este sería el encargo:

```
Con 'pacientes.csv' (cohorte SINTÉTICA; target de clasificación: evento_cv,
prevalencia ≈19%), en español y por celdas, actúa como mi copiloto de ciencia de
datos y trabaja EN BUCLE:
1. Primero critica mi encargo (metaprompting): dime qué te falta y propónme la
   métrica y la validación (por paciente, sin fuga) con su porqué.
2. Construye un baseline y una regresión logística; mídelas con AUC y con la
   sensibilidad en el grupo que citaría; enséñame la curva de calibración.
3. Prueba Random Forest y un boosting; compáralos con la logística en validación
   cruzada. Dime qué gana y por qué (¿el riesgo es log-aditivo?).
4. Interpreta clínicamente: efecto del tabaquismo (nunca/ex/activo), del HDL y de
   la actividad física. ¿Tiene sentido?
5. Con el mejor modelo, ajusta hiperparámetros SIN tocar el test; elige umbral
   según el coste del falso negativo; evalúa UNA vez en el holdout y entrégame un
   mini-informe con las decisiones y su porqué.
Párate a pedir mi visto bueno en la métrica final, el modelo elegido y el umbral.
```

Con esto cierras el arco de todo el curso: sabes **qué** modelo encaja en cada problema (U2-U8), **dónde** están los modelos ya hechos (U9) y ahora **cómo dirigir** el proceso entero como un copiloto de ciencia de datos. Solo queda una pieza transversal y decisiva —la ética, el sesgo, la validación clínica y la privacidad—, que es el terreno de la Unidad 11.
